# NB69: Gaming Telemetry

Kafka -> Spark -> Redis/Cassandra

## 1. Environment Setup

Installs **Java 8**, **Spark 3.5.0**, **Kafka 3.6.1**, and Python libraries (PySpark, Kafka-Python, Redis, Mongo, ES, Cassandra, MinIO).

In [ ]:
# Install Dependencies (Java 8, Spark 3.5.0, Kafka 3.6.1)
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
!wget -q https://archive.apache.org/dist/spark/spark-3.5.0/spark-3.5.0-bin-hadoop3.tgz
!tar xf spark-3.5.0-bin-hadoop3.tgz
!wget -q https://archive.apache.org/dist/kafka/3.6.1/kafka_2.13-3.6.1.tgz
!tar xf kafka_2.13-3.6.1.tgz
!pip uninstall -y numpy
!pip install -q "numpy<2.0.0"
!pip install -q findspark pyspark kafka-python redis pymongo elasticsearch==7.10.1 cassandra-driver minio

# Environment Variables
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.5.0-bin-hadoop3"
import findspark
findspark.init()

## 2. Start Services

Starts background services needed for this pipeline:
- **Kafka** (Zookeeper + Broker)
- **Redis**
- **Cassandra**

In [ ]:
# Start Kafka
!./kafka_2.13-3.6.1/bin/zookeeper-server-start.sh -daemon ./kafka_2.13-3.6.1/config/zookeeper.properties
!./kafka_2.13-3.6.1/bin/kafka-server-start.sh -daemon ./kafka_2.13-3.6.1/config/server.properties
# Start Redis
!apt-get install redis-server -qq > /dev/null
!service redis-server start
# Start Cassandra
!wget -q https://archive.apache.org/dist/cassandra/4.1.3/apache-cassandra-4.1.3-bin.tar.gz
!tar xf apache-cassandra-4.1.3-bin.tar.gz
!apache-cassandra-4.1.3/bin/cassandra -R > cassandra.log 2>&1 &

import time, socket, os
def wait_for_port(port, host='localhost', timeout=120):
    start_time = time.time()
    while True:
        try:
            with socket.create_connection((host, port), timeout=1):
                print(f"Service at {host}:{port} is ready!")
                return True
        except (OSError, ConnectionRefusedError):
            if time.time() - start_time > timeout:
                print(f"Timeout waiting for {host}:{port} to start.")
                # Dump logs for debugging
                if os.path.exists('minio.log'):
                    print('--- MINIO LOG ---')
                    print(open('minio.log').read())
                if os.path.exists('es.log'):
                    print('--- ES LOG ---')
                    print(open('es.log').read())
                if os.path.exists('cassandra.log'):
                    print('--- CASSANDRA LOG ---')
                    print(open('cassandra.log').read())
                raise Exception(f"Service at {host}:{port} failed to start.")
            time.sleep(2)

# Wait for services
wait_for_port(9092) # Kafka
wait_for_port(9042) # Cassandra
time.sleep(10) # Extra buffer for Cassandra
wait_for_port(6379) # Redis


## 3. Create Kafka Topic

Creates a topic named `input-topic`.

In [ ]:
# Create Topic
!./kafka_2.13-3.6.1/bin/kafka-topics.sh --create --topic input-topic --bootstrap-server localhost:9092 --replication-factor 1 --partitions 1

## 4. Producer (Game Events)

Simulates headshots.

In [ ]:
from kafka import KafkaProducer
import json, time, random
print("Starting Game Event Producer...")
producer = KafkaProducer(bootstrap_servers='localhost:9092')
print("Sending 500 events...")
for _ in range(500):
    data = {'player': f'p{random.randint(1,100)}', 'action': 'headshot'}
    producer.send('input-topic', json.dumps(data).encode('utf-8'))
producer.flush()
print("Producer finished.")

## 5. Anti-Cheat Engine

1. Increments headshot count in Redis.
2. If count > Threshold, adds player to `ban_list` set.

In [ ]:
%%writefile kafka_consumer.py
from pyspark.sql import SparkSession
import redis
import json

spark = SparkSession.builder.appName("Gaming").getOrCreate()

def process_batch(df, epoch_id):
    rows = df.collect()
    r = redis.Redis()
    for row in rows:
        evt = json.loads(row.value)
        count = r.incr(f"hs:{evt['player']}")
        if count > 10: # Threshold
             r.sadd("ban_list", evt['player'])
             print(f"BANNED {evt['player']}")
    print(f"Batch {epoch_id} processed.")

print("Starting Spark Streaming Job...")
df = spark.readStream.format("kafka").option("kafka.bootstrap.servers", "localhost:9092").option("subscribe", "input-topic").option("startingOffsets", "earliest").load()
query = df.selectExpr("CAST(value AS STRING)").writeStream.foreachBatch(process_batch).start()
query.awaitTermination(30)
print("Spark Job Finished.")

In [ ]:
!spark-submit --packages org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0 kafka_consumer.py

## 6. Verification

Check Ban List in Redis.

In [ ]:
import redis
r = redis.Redis()
banned = r.smembers("ban_list")
print(f"Banned Players: {banned}")